# Counting vehicles by class for signal timing (UVH-26, YOLOv8)

AIML ZG525 Computer Vision, EC-1 Assignment 1 · Track B (object detection)

This notebook runs top to bottom on a free Colab T4 (*Runtime → Change runtime type → T4 GPU*).

- **`RUN_TRAINING`** (next cell): `True` retrains both models; `False` skips training and evaluates the weights saved on Drive.
- The dataset is downloaded by the notebook from the Hugging Face Hub at a pinned revision; no data files are embedded.
- Sections follow the report's 8 steps. Every number quoted in the report appears in a cell output below.
- Optional Colab secrets: `HF_TOKEN` (faster Hub downloads), `GITHUB_TOKEN` (only if the repo is private).

In [ ]:
# ---- Run settings: the only cell a grader should need to edit ----
RUN_TRAINING = True   # False: skip training, evaluate the weights saved under DRIVE_ROOT/runs
REPO_URL = "https://github.com/<github-user>/uvh26-pcu-counting.git"
COMMIT = "main"       # pinned to a commit hash for the final submission run
DRIVE_ROOT = "/content/drive/MyDrive/uvh26"

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    repo_dir = Path("/content/uvh26-pcu-counting")
    if not repo_dir.exists():
        try:
            token = userdata.get("GITHUB_TOKEN")  # only needed for a private repo
        except Exception:
            token = None
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(f"git clone -q {url} {repo_dir}", shell=True, check=True)
    os.chdir(repo_dir)
    sh(f"git fetch -q && git checkout -q {COMMIT}")
    # Keep Colab's preinstalled torch/opencv/numpy (CUDA-matched, no restart); pin the rest.
    sh("grep -v colab-skip requirements.txt > /tmp/req-colab.txt && pip install -q -r /tmp/req-colab.txt")
    os.environ.update({
        "UVH_RAW_ROOT": "/content/raw",              # original PNGs on fast local disk
        "UVH_DATA_ROOT": f"{DRIVE_ROOT}/prepared",   # JPEGs + labels survive a disconnect
        "UVH_RUNS_ROOT": f"{DRIVE_ROOT}/runs",
        "UVH_WEIGHTS_DIR": f"{DRIVE_ROOT}/weights",
    })
else:
    repo_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(repo_dir)

sys.path.insert(0, str(repo_dir))
commit = subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("repo:", repo_dir, "| commit:", commit or "(no commits yet)")

In [ ]:
import torch
from src.common import library_versions, load_config, set_seed

cfg = load_config()
set_seed(cfg["seed"])
print("seed:", cfg["seed"])
for name, version in library_versions().items():
    print(f"  {name:16s} {version}")
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("paths:", {k: str(v) for k, v in cfg["paths"].items()})

## 1. Problem and task type

*Written with the report.* Signal timing needs per-class vehicle **counts** at a junction, converted to PCU. Counting needs instances, so this is object detection: classification cannot count, and segmentation adds labelling and compute cost without changing the count.

## 2. Data and split  *(Phase 1)*

UVH-26 (IISc AIM with Bengaluru Traffic Police, CC BY 4.0), pinned Hub revision in `configs/base.yaml`. Only the annotation JSONs and a sampled image pool are downloaded.

In [ ]:
from src import download

ann_paths = download.download_annotations(cfg)
index = download.build_file_index(cfg)
print(index.groupby(["official_split", "subfolder"]).size().rename("images"))

## 3. Preprocessing and augmentation  *(Phases 1-2)*

## 4. Baseline: YOLOv8 fine-tuned on MV labels  *(Phase 2)*

## 5. Ablation: MV vs STAPLE training labels  *(Phase 3)*

## 6. Evaluation  *(Phase 4 threshold on val; Phase 5 sealed test, once)*

## 7. Error analysis  *(Phase 6)*

## 8. Conclusion